# Gestire le mail con Python

Gli esempi che usano la rete richiedono un server SMTP/IMAP locale e una
webmail. Questo notebook si basa sull'uso di
[GreenMail](https://greenmail-mail-test.github.io/greenmail/) (server email
in-memory per il testing) e [Roundcube](https://roundcube.net/) (client
webmail), avviabili con `docker compose up` dalla directory corrennte.

## Reperire le credenziali

Per prima cosa reperiamo le credenziali per accedere al server SMTP/IMAP e alla
webmail dail file di configurazione `.env` che è usato da `docker-compose.yml`.

In [1]:
import pathlib

users_str = next(
    l.split('=', 1)[1]
    for l in pathlib.Path('.env').read_text().splitlines()
    if l.startswith('GREENMAIL_USERS=')
)

sender_ap, recipient_ap = users_str.split(',')

def ap(c):
  up, d = c.split('@')
  u, p = up.split(':')
  return f'{u}@{d}', p

sender, sender_pass = ap(sender_ap)
recipient, recipient_pass = ap(recipient_ap)

## Comporre le email

La libreria standard di Python include un pacchetto
[`email`](https://docs.python.org/3/library/email.html) per comporre, mandare e
leggere le mail (senza coinvolgere la rete).

Ci sono due generazioni di API:

| API | Class | Policy | Notes |
|-----|-------|--------|-------|
| Legacy (< 3.6) | `email.message.Message` | `compat32` | Low-level, gestione dell'encoding fragile |
| **Modern (≥ 3.6)** | **`email.message.EmailMessage`** | **`default` / `EmailPolicy`** | High-level, basata su Unicode, raccomandata |

In [2]:
from email.message import EmailMessage


### Mail semplice

In [3]:
simple_msg = EmailMessage()
simple_msg['Subject'] = 'A simple email'
simple_msg['From'] = sender
simple_msg['To'] = recipient
simple_msg.set_content('This is just some text.')

print(simple_msg)

Subject: A simple email
From: snd@foo.bar
To: rec@goo.bar
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: 7bit
MIME-Version: 1.0

This is just some text.



### Con una alternativa HTML

In [4]:
html_body = """\
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>"""

html_msg = EmailMessage()
html_msg['Subject'] = 'An HTML email'
html_msg['From'] = sender
html_msg['To'] = recipient
html_msg.set_content('Plain text fallback.')
html_msg.add_alternative(html_body, subtype='html')

print(html_msg)

Subject: An HTML email
From: snd@foo.bar
To: rec@goo.bar
MIME-Version: 1.0
Content-Type: multipart/alternative;
 boundary="===============8725674482526225643=="

--===============8725674482526225643==
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: 7bit

Plain text fallback.

--===============8725674482526225643==
Content-Type: text/html; charset="utf-8"
Content-Transfer-Encoding: 7bit
MIME-Version: 1.0

<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

--===============8725674482526225643==--



### Con un allegato

In [5]:
csv_data = "name,score\nAlice,95\nBob,87\nCarol,92\n"

attachment_msg = EmailMessage()
attachment_msg['Subject'] = 'A mail with an attachment'
attachment_msg['From'] = sender
attachment_msg['To'] = recipient
attachment_msg.set_content('Please find the results attached.')
attachment_msg.add_attachment(csv_data.encode(), maintype='text', subtype='csv', filename='results.csv')

print(attachment_msg)

Subject: A mail with an attachment
From: snd@foo.bar
To: rec@goo.bar
MIME-Version: 1.0
Content-Type: multipart/mixed; boundary="===============8606272989935130980=="

--===============8606272989935130980==
Content-Type: text/plain; charset="utf-8"
Content-Transfer-Encoding: 7bit

Please find the results attached.

--===============8606272989935130980==
Content-Type: text/csv
Content-Transfer-Encoding: base64
Content-Disposition: attachment; filename="results.csv"
MIME-Version: 1.0

bmFtZSxzY29yZQpBbGljZSw5NQpCb2IsODcKQ2Fyb2wsOTIK

--===============8606272989935130980==--



### Serializzare e deserializzare i messaggi

In [6]:
import email

# come stringa

raw_str = str(html_msg)

raw_str[:100]


'Subject: An HTML email\nFrom: snd@foo.bar\nTo: rec@goo.bar\nMIME-Version: 1.0\nContent-Type: multipart/a'

In [7]:
# come byte

raw_bytes = html_msg.as_bytes()    # bytes — what actually goes on the wire

raw_bytes[:100]

b'Subject: An HTML email\nFrom: snd@foo.bar\nTo: rec@goo.bar\nMIME-Version: 1.0\nContent-Type: multipart/a'

In [8]:
# Deserializzare 

parsed = email.message_from_bytes(raw_bytes, policy=email.policy.default)

print(f'Content-Type: {parsed.get_content_type()}')
print(f'Subject: {parsed["Subject"]}')


Content-Type: multipart/alternative
Subject: An HTML email


### Esaminare le parti

In [9]:
for part in parsed.walk():
  ct = part.get_content_type()
  cd = part.get_content_disposition()
  print(f'part: {ct}, disposition={cd}')

part: multipart/alternative, disposition=None
part: text/plain, disposition=None
part: text/html, disposition=None


## Inviare le mail con SMTP

Per inviare le mail, usiamo il modulo [`smtplib`](https://docs.python.org/3/library/smtplib.html) della libreria standard.

In [10]:
from smtplib import SMTP

with SMTP('localhost', 3025) as smtp:
    smtp.set_debuglevel(1) # questo consente di ispezionare il protocollo
    smtp.login(sender, sender_pass)
    smtp.send_message(simple_msg)
    smtp.send_message(html_msg)
    smtp.send_message(attachment_msg)

send: 'ehlo [127.0.1.1]\r\n'
reply: b'250-/172.19.0.2\r\n'
reply: b'250 AUTH PLAIN LOGIN XOAUTH2\r\n'
reply: retcode (250); Msg: b'/172.19.0.2\nAUTH PLAIN LOGIN XOAUTH2'
send: 'AUTH PLAIN AHNuZEBmb28uYmFyAHNuZHA=\r\n'
reply: b'235 2.7.0  Authentication Succeeded\r\n'
reply: retcode (235); Msg: b'2.7.0  Authentication Succeeded'
send: 'mail FROM:<snd@foo.bar>\r\n'
reply: b'250 OK\r\n'
reply: retcode (250); Msg: b'OK'
send: 'rcpt TO:<rec@goo.bar>\r\n'
reply: b'250 OK\r\n'
reply: retcode (250); Msg: b'OK'
send: 'data\r\n'
reply: b'354 Start mail input; end with <CRLF>.<CRLF>\r\n'
reply: retcode (354); Msg: b'Start mail input; end with <CRLF>.<CRLF>'
data: (354, b'Start mail input; end with <CRLF>.<CRLF>')
send: b'Subject: A simple email\r\nFrom: snd@foo.bar\r\nTo: rec@goo.bar\r\nContent-Type: text/plain; charset="utf-8"\r\nContent-Transfer-Encoding: 7bit\r\nMIME-Version: 1.0\r\n\r\nThis is just some text.\r\n.\r\n'
reply: b'250 OK\r\n'
reply: retcode (250); Msg: b'OK'
data: (250, b'OK')
s

## Ricevere la mail con IMAP

### Usando la libreria standard

In [11]:
import imaplib, email as emaillib

with imaplib.IMAP4('localhost', 3143) as imap:
  imap.login(recipient, recipient_pass)
  imap.select('INBOX')
  _, msg_nums = imap.search(None, 'ALL')
  for num in msg_nums[0].split():
    _, data = imap.fetch(num, '(RFC822)')
    msg = emaillib.message_from_bytes(data[0][1])
    print(f"Subject: {msg['Subject']}")
    for part in msg.walk():
      ct = part.get_content_type()
      cd = part.get_content_disposition()
      if cd == 'attachment':
        print(f'  attachment: {part.get_filename()}')
        print(part.get_payload(decode=True).decode())
      elif ct == 'text/html':
        print('  html alternative:')
        print(part.get_payload(decode=True).decode())


Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body><h2>Hello!</h2></body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

S

### Usando imap-tools


`imaplib` is complete but low-level: responses are raw bytes, search criteria are stringly-typed, and you have to parse everything yourself. The third-party [`imap-tools`](https://github.com/ikvk/imap_tools) library wraps it with a clean, Pythonic API.

| Task | `imaplib` | `imap-tools` |
|------|-----------|--------------|
| Fetch all messages | `search` + `fetch` loop | `mailbox.fetch()` iterator |
| Read subject | `email.message_from_bytes(raw)['Subject']` | `msg.subject` |
| List attachments | walk parts, filter by disposition | `msg.attachments` |
| Filter server-side | string criteria | `AND(from_='alice@…', seen=False)` |

In [12]:
from imap_tools import MailBoxUnencrypted, AND

with MailBoxUnencrypted('localhost', 3143).login(recipient, recipient_pass) as mailbox:
  for msg in mailbox.fetch(AND(from_='snd@foo.bar')):
    print(f'Subject: {msg.subject}')
    for att in msg.attachments:
      print(f'  attachment: {att.filename}')
      print(att.payload.decode())
    if msg.html:
      print('  html alternative:')
      print(msg.html)


Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body><h2>Hello!</h2></body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

Subject: A simple email
Subject: An HTML email
  html alternative:
<html><body>
<h2>Hello!</h2>
<p>This message has a <strong>rich HTML</strong> body.</p>
</body></html>

Subject: A mail with an attachment
  attachment: results.csv
name,score
Alice,95
Bob,87
Carol,92

S